# Retrieval-Augmented Generation (RAG) for Research Documents

**MIDAS AI in Research Handbook — RAG for Research Documents**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/rag_demo.ipynb)

---

This notebook builds a complete retrieval-augmented generation (RAG) pipeline from scratch. By the end, you will have a working system that embeds a collection of documents, retrieves relevant passages for any question, and generates a grounded answer based only on what it finds.

**What you need:** A Google account to run in Colab. No API keys, no external services, no local installation required.

**Data:** A small collection of synthetic research methodology excerpts — these are fictional example documents written for this tutorial, covering environmental science, public health, and economics. You can swap them out for your own documents at the end.

## Step 1 — Install Dependencies

We need two libraries: `sentence-transformers` for creating embeddings and `transformers` for the generation step. Both are open-source and run on CPU, so no GPU is required.

In [ ]:
!pip install -q sentence-transformers transformers 2>&1 | tail -3
print("Installation complete.")

## Step 2 — Import Libraries

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline
import textwrap

print("Libraries loaded.")

## Step 3 — Load Sample Documents

A RAG system starts with a collection of documents. In a real project these would be your PDFs, interview transcripts, lab protocols, or literature notes. Here we use 15 short synthetic excerpts written for this tutorial. Each one represents the kind of text you might find in a research project's shared documentation.

**Important:** These are fictional example documents. They are not copied from any published source.

In [ ]:
# Each document is a dictionary with a source label and the text content.
# In a real project, you would load these from files on your machine.

DOCUMENTS = [
    {
        "source": "IRB Protocol v3.2 — Section 4: Participant Confidentiality",
        "text": (
            "All audio recordings of participant interviews will be stored on a password-protected "
            "university server. Recordings will be transcribed by a trained research assistant who "
            "has completed HIPAA training and signed a data confidentiality agreement. "
            "Transcripts will be de-identified by replacing all names, locations, and "
            "identifying details with generic placeholders (e.g., [PARTICIPANT], [CITY]) "
            "before being shared with the broader research team. "
            "No raw recordings or identified transcripts will be uploaded to external cloud services."
        )
    },
    {
        "source": "Coding Manual v2.1 — Economic Hardship Indicators",
        "text": (
            "Code ECON_HARDSHIP when a participant describes difficulty meeting basic material needs. "
            "This includes mentions of being unable to pay rent or utilities, skipping meals, "
            "borrowing money from family or friends to cover essential expenses, or choosing "
            "between competing necessities such as medication and food. "
            "Apply this code even when the participant describes the difficulty in the past tense "
            "or frames it indirectly (e.g., 'we had a rough year'). "
            "Do not apply ECON_HARDSHIP for general statements about low income or poverty "
            "without a specific hardship event being described."
        )
    },
    {
        "source": "Coding Manual v2.1 — Social Support Networks",
        "text": (
            "Code SUPPORT_FORMAL for references to institutional support: government benefits, "
            "nonprofit services, food banks, Medicaid, housing assistance programs, or any "
            "organized service provided by an agency rather than a personal relationship. "
            "Code SUPPORT_INFORMAL for support received from family, friends, neighbors, "
            "or religious community members acting in a personal capacity. "
            "A participant may describe receiving both types in the same passage; "
            "apply both codes when this occurs. "
            "If the source of support is ambiguous, flag the passage for team review "
            "rather than making an independent judgment."
        )
    },
    {
        "source": "Methodological Decision Log — Wave 2 Data Collection",
        "text": (
            "Decision (2023-11-14): After reviewing Wave 1 attrition rates, the team agreed to "
            "add a $25 incentive for Wave 2 participation. The concern raised by team members "
            "was that differential incentives across waves could introduce selection effects "
            "if participants who returned were systematically different from those who did not. "
            "We reviewed existing literature on longitudinal incentive design and concluded "
            "that the attrition risk outweighed the selection concern at this sample size. "
            "This decision should be documented in the methods section of any publication "
            "using Wave 2 data."
        )
    },
    {
        "source": "Interview Administration Protocol — Consent Procedures",
        "text": (
            "Before beginning the recorded portion of the interview, the research assistant "
            "must read the consent form aloud with the participant, pausing after each section "
            "to invite questions. Do not rush through the consent process even if the participant "
            "indicates they have read the form. The participant must verbally confirm they "
            "understand their right to withdraw at any time without penalty before recording begins. "
            "If a participant expresses hesitation about audio recording, offer the option of "
            "handwritten notes only. Document this in the field notes log."
        )
    },
    {
        "source": "Literature Review Notes — Air Quality and Health Outcomes",
        "text": (
            "A growing body of evidence links long-term exposure to particulate matter (PM2.5) "
            "with cardiovascular and respiratory disease burden in urban populations. "
            "The mechanisms proposed in recent work include systemic inflammation triggered "
            "by inhaled particles, oxidative stress in lung tissue, and indirect pathways "
            "through disrupted sleep from nighttime noise associated with traffic sources. "
            "Importantly, health effects are not distributed evenly: lower-income neighborhoods "
            "and communities of color consistently bear higher exposure levels, "
            "making air quality an environmental justice issue as well as a public health one."
        )
    },
    {
        "source": "Literature Review Notes — Carbon Pricing Policy",
        "text": (
            "Carbon pricing mechanisms fall into two broad categories: carbon taxes, "
            "which set a price per unit of emissions and let the market determine the quantity reduced, "
            "and cap-and-trade systems, which set a quantity limit and let markets determine the price. "
            "Economic theory predicts similar efficiency outcomes under either approach, "
            "but empirical evidence from implemented programs suggests that political economy "
            "factors — particularly the visibility of price increases to consumers — "
            "make carbon taxes more politically difficult to maintain at high levels over time. "
            "British Columbia's revenue-neutral carbon tax is frequently cited as the best-studied "
            "real-world example of a carbon pricing program with measurable emission reductions."
        )
    },
    {
        "source": "Field Notes — Site Visit, Rural County Health Department",
        "text": (
            "The health department building is a converted county office, shared with "
            "the zoning board and a WIC office. The waiting area had approximately twelve people "
            "when we arrived at 9 AM. Staff mentioned that the primary care provider "
            "visits only on Tuesdays and Thursdays; on other days, the nurse practitioner "
            "handles most clinical questions. Several participants later mentioned "
            "this limited availability when asked about barriers to accessing healthcare. "
            "Transportation came up repeatedly: the nearest hospital is 47 miles away, "
            "and public transit does not serve rural routes in the county."
        )
    },
    {
        "source": "Data Preparation Notes — Missing Data Strategy",
        "text": (
            "For continuous covariates with missingness below 10%, we will use "
            "multiple imputation by chained equations (MICE), retaining five imputed datasets "
            "and pooling estimates using Rubin's rules. "
            "Variables with missingness above 10% will be examined individually "
            "before a decision is made: if the missingness mechanism is plausibly "
            "missing at random (MAR), MICE will still apply; if it appears to be "
            "informative (e.g., participants who refused to report income), "
            "we will model the missingness explicitly as a sensitivity analysis. "
            "Binary outcomes with any missing values will be analyzed under "
            "both complete case and MICE assumptions and results compared."
        )
    },
    {
        "source": "Lab Onboarding Guide — File Naming Conventions",
        "text": (
            "All data files must follow the naming convention: "
            "[ProjectCode]_[DataType]_[WaveNumber]_[YYYYMMDD]_[Initials].ext "
            "For example: HRST_Transcript_W2_20231105_JL.docx. "
            "Never use spaces in file names; use underscores. "
            "Do not include version numbers in the file name itself; "
            "version control is handled through our shared Box folder's version history. "
            "Raw data files should never be edited in place. "
            "If corrections are needed, create a cleaned copy in the designated "
            "processed data subfolder and leave the original untouched."
        )
    },
    {
        "source": "Literature Review Notes — Longitudinal Survey Attrition",
        "text": (
            "Survey attrition in longitudinal studies is rarely random. "
            "Participants who drop out tend to differ from those who remain "
            "on characteristics that are often directly relevant to the study's outcomes — "
            "economic instability, residential mobility, and poor health are among the "
            "most commonly documented predictors of attrition in social science panels. "
            "This means that complete-case analyses, which use only participants "
            "who completed all waves, are likely to produce biased estimates "
            "in precisely the domains the research is designed to study. "
            "Analysts should report attrition patterns explicitly and conduct "
            "sensitivity analyses under plausible missing data assumptions."
        )
    },
    {
        "source": "Interview Administration Protocol — Field Safety Guidelines",
        "text": (
            "Research assistants must check in with the lab coordinator before "
            "leaving for any off-campus interview site and again upon returning. "
            "If an interview is conducted in a participant's home, a second team member "
            "should be notified of the address and expected return time. "
            "If a participant discloses an immediate safety concern during the interview — "
            "domestic violence, suicidal ideation, or harm to a child — "
            "the research assistant should follow the mandated reporting procedures "
            "outlined in Appendix B of the IRB protocol. "
            "Discontinue the interview if you feel unsafe at any point; "
            "the data is not worth the risk."
        )
    },
    {
        "source": "Methodological Decision Log — Qualitative Sampling Strategy",
        "text": (
            "Decision (2024-02-03): The team agreed to move from a purposive sampling frame "
            "to a combination of purposive and snowball sampling for the second phase. "
            "Purposive sampling alone was not reaching participants with active substance use, "
            "who were underrepresented in wave one relative to their prevalence in the target population. "
            "Snowball recruitment through community health workers has been approved "
            "under amendment 4 to the IRB protocol. "
            "We will track and report recruitment pathway (purposive vs. snowball) "
            "for all wave-two participants to allow sensitivity analysis by recruitment method."
        )
    },
    {
        "source": "Literature Review Notes — Water Contamination and Environmental Justice",
        "text": (
            "Research on drinking water contamination in the United States consistently finds "
            "that communities with lower median incomes and higher proportions of racial "
            "and ethnic minorities face greater exposure to water quality violations. "
            "This pattern holds across multiple contaminants including lead, nitrates, "
            "and disinfection byproducts, and persists after controlling for system size "
            "and ownership type. Enforcement patterns compound the disparity: "
            "violations in disadvantaged communities take longer to remediate "
            "and result in fewer formal sanctions against the water system operator."
        )
    },
    {
        "source": "Analysis Plan — Primary Outcomes",
        "text": (
            "The primary outcome for Aim 1 is six-month housing stability, "
            "operationalized as remaining in the same residence from baseline to the six-month follow-up. "
            "This will be modeled using logistic regression with robust standard errors, "
            "adjusting for baseline age, gender, income-to-poverty ratio, and prior eviction history. "
            "For Aim 2, the primary outcome is self-reported mental health status "
            "at twelve months, measured with the PHQ-9. We will use linear regression "
            "for this outcome given the near-continuous distribution of PHQ-9 scores "
            "in our pilot data. All models will be estimated using the pooled "
            "imputed datasets following Rubin's rules."
        )
    }
]

print(f"Loaded {len(DOCUMENTS)} documents.")
for i, doc in enumerate(DOCUMENTS):
    print(f"  {i+1:2d}. {doc['source'][:60]}...")


## Step 4 — Chunk the Documents

A language model can only process a limited amount of text at once, and retrieving an entire long document in response to a specific question is inefficient. We split each document into smaller pieces called **chunks** — short enough to be specific, long enough to carry meaning.

For these short example documents, each document is already a single chunk. For longer documents like full interview transcripts or multi-page protocols, you would split by paragraph, by a fixed character count, or at natural section boundaries. The notebook's 'Try This' section shows how to handle longer texts.

In [ ]:
def chunk_documents(docs, max_chars=800, overlap=80):
    """
    Split documents into chunks. For short documents that fit in one chunk,
    returns them as-is. For longer documents, splits with a small overlap
    so that context at chunk boundaries is not lost.
    """
    chunks = []
    for doc in docs:
        text = doc["text"]
        source = doc["source"]
        if len(text) <= max_chars:
            chunks.append({"text": text, "source": source})
        else:
            start = 0
            while start < len(text):
                end = min(start + max_chars, len(text))
                chunk_text = text[start:end]
                chunks.append({"text": chunk_text, "source": source})
                if end == len(text):
                    break
                start += max_chars - overlap
    return chunks

chunks = chunk_documents(DOCUMENTS)
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {np.mean([len(c['text']) for c in chunks]):.0f} characters")

## Step 5 — Create Embeddings

We use `all-MiniLM-L6-v2` from the `sentence-transformers` library to convert each chunk into a 384-dimensional vector. This is the same library covered in the NLP with BERT chapter. The model is small enough to run quickly on CPU and produces embeddings that work well for semantic similarity tasks.

This step runs once. In a production system, you would save the embeddings to disk so you do not need to re-embed every time the system starts.

In [ ]:
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]

print("Embedding documents...")
chunk_embeddings = embed_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

print(f"\nEmbedding matrix shape: {chunk_embeddings.shape}")
print("Each chunk is now represented as a vector of",
      chunk_embeddings.shape[1], "numbers.")

## Step 6 — Build the Retrieval Function

Retrieval works by comparing your question's embedding to every chunk's embedding and returning the closest matches. We use cosine similarity, which measures the angle between two vectors rather than the distance between their endpoints. Because we normalized the embeddings in Step 5, this is equivalent to a dot product, which numpy computes very quickly.

Cosine similarity ranges from -1 to 1. A score above 0.4 generally indicates meaningful semantic overlap; a score above 0.6 is a strong match. You will see these numbers directly in the output below.

In [ ]:
def retrieve(query, top_k=3, min_score=0.0):
    """
    Find the top_k most relevant chunks for a given query.
    Returns a list of (chunk_dict, score) tuples.
    """
    query_vec = embed_model.encode([query], normalize_embeddings=True)
    # Dot product of normalized vectors = cosine similarity
    scores = np.dot(chunk_embeddings, query_vec.T).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        if scores[idx] >= min_score:
            results.append((chunks[idx], float(scores[idx])))
    return results

# Quick test
print("Testing retrieval with a sample query...\n")
test_results = retrieve("how should we handle missing data")
for chunk, score in test_results:
    print(f"Score: {score:.3f}  |  Source: {chunk['source'][:55]}...")
    print(textwrap.fill(chunk['text'][:200] + '...', width=80, initial_indent='  '))
    print()

## Step 7 — Explore Retrieval Quality

Before adding a generative model, it is worth spending time with retrieval alone. The retrieval step is where most RAG systems fail: if the wrong passages come back, no language model can produce a correct answer from them.

Try the queries below and look at what comes back. Ask yourself whether each returned passage genuinely contains information relevant to the question.

In [ ]:
sample_queries = [
    "what are the rules for audio recording consent",
    "how do we code mentions of indirect financial assistance",
    "can transcripts be uploaded to external platforms",
    "what is the file naming convention for data files",
    "what does the literature say about water contamination",
]

for q in sample_queries:
    print("=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)
    results = retrieve(q, top_k=2)
    for i, (chunk, score) in enumerate(results):
        print(f"\nResult {i+1}  (score: {score:.3f})")
        print(f"Source: {chunk['source']}")
        print(textwrap.fill(chunk['text'], width=75, initial_indent='  '))
    print()

## Step 8 — Add Generation

Now we add the second half of RAG: a language model that reads the retrieved passages and composes a response based on them.

We use `google/flan-t5-base`, a small open-weight model from Google that runs on CPU and works well for focused question-answering tasks given a context. Loading it takes about a minute on a fresh Colab session.

The key design decision here is the prompt template. We explicitly tell the model to answer only from the provided context. This is what keeps the system grounded in your documents rather than drawing on the model's general training data.

In [ ]:
print("Loading generative model (this may take 1-2 minutes)...")

from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

_model_name = "google/flan-t5-base"
_tokenizer = T5Tokenizer.from_pretrained(_model_name)
_model = T5ForConditionalGeneration.from_pretrained(_model_name)

def generator(prompt, max_new_tokens=200):
    """Wraps FLAN-T5 to match the pipeline interface used below."""
    text = prompt if isinstance(prompt, str) else prompt[0]
    inputs = _tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = _model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated = _tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": generated}]

print("Generative model loaded.")

In [ ]:
def format_context(retrieved_results):
    """Format retrieved chunks into a context block for the prompt."""
    parts = []
    for i, (chunk, score) in enumerate(retrieved_results):
        parts.append(f"[Source: {chunk['source']}]\n{chunk['text']}")
    return "\n\n".join(parts)

def rag(question, top_k=3, verbose=True):
    """
    Full RAG pipeline: retrieve relevant chunks, then generate a grounded answer.
    """
    # Step 1: Retrieve
    results = retrieve(question, top_k=top_k)
    context = format_context(results)

    # Step 2: Build prompt
    prompt = (
        f"Answer the question using only the information in the context below. "
        f"If the context does not contain enough information to answer, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )

    # Step 3: Generate
    output = generator(prompt)[0]["generated_text"].strip()

    if verbose:
        print("QUESTION:", question)
        print("-" * 60)
        print("RETRIEVED SOURCES:")
        for chunk, score in results:
            print(f"  [{score:.3f}] {chunk['source'][:65]}")
        print("-" * 60)
        print("GENERATED ANSWER:")
        print(textwrap.fill(output, width=75))
        print()
    return output, results

print("RAG pipeline ready.")

## Step 9 — Run the Full Pipeline

Now we can ask questions and get grounded answers. Notice that the output tells you exactly which source documents were used, which means you can always trace a claim back to its origin.

In [ ]:
questions = [
    "What are the requirements before starting an audio recording with a participant?",
    "How should I handle missing data when more than 10 percent of values are absent?",
    "What does ECON_HARDSHIP mean and when should I apply it?",
    "What does the literature say about who is most affected by air quality problems?",
]

for q in questions:
    print("=" * 70)
    rag(q)
    print()

## Step 10 — Try Your Own Documents

Replace the contents of `MY_DOCUMENTS` below with excerpts from your own materials. Good candidates for a first test are documents that are not sensitive: a literature review section, a methods chapter, or some project notes.

Keep each excerpt between 100 and 800 characters for best results. For longer documents, the chunking function in Step 4 will split them automatically.

In [ ]:
MY_DOCUMENTS = [
    {
        "source": "Your document title or filename here",
        "text": (
            "Paste your document text here. "
            "This can be a few sentences or a few paragraphs."
        )
    },
    {
        "source": "Second document",
        "text": "Another excerpt from a different source."
    }
]

# To activate your documents:
# 1. Replace the text above with your own content
# 2. Run this cell
# 3. Run the next cell to rebuild the index

print(f"You defined {len(MY_DOCUMENTS)} custom document(s).")

In [ ]:
# Rebuild the index with your documents
# (comment this cell out to keep using the original sample documents)

my_chunks = chunk_documents(MY_DOCUMENTS)
my_texts = [c["text"] for c in my_chunks]
my_embeddings = embed_model.encode(my_texts, normalize_embeddings=True, show_progress_bar=True)

# Temporarily swap in your documents
orig_chunks, orig_embeddings = chunks, chunk_embeddings
chunks, chunk_embeddings = my_chunks, my_embeddings

print(f"Index rebuilt with {len(my_chunks)} chunk(s) from your documents.")
print("Run the rag() function with your own questions below.")

# Example:
# rag("paste your question here")

## What to Do Next

This notebook demonstrates the core RAG loop on a small document collection. If you want to take this further:

- For larger corpora (hundreds of documents or more), look into [ChromaDB](https://www.trychroma.com/) or [FAISS](https://github.com/facebookresearch/faiss) as alternatives to the numpy index used here. They store vectors on disk and support much larger collections efficiently.

- For better answers on complex questions, replacing `flan-t5-base` with a larger model will help substantially. If you have access to a GPU on Great Lakes or another HPC cluster, a 7-billion-parameter instruction-tuned model is a practical option.

- For documents that cannot leave a university-approved server, the entire pipeline above runs on Great Lakes with no modification. None of the models used here require an internet connection once downloaded, and no data is sent to any external service during retrieval or generation.

- For sensitive data at U-M, check with your IRB and data governance office about whether Maizey (https://its.umich.edu/computing/ai/maizey) covers your use case before building a custom pipeline.